# Fine-tune Invoice Parser

## Dataset: [mychen76/invoices-and-receipts_ocr_v1](https://huggingface.co/datasets/mychen76/invoices-and-receipts_ocr_v1/)

### 1.Gather Invoice Image/Label Pairs

!pip install datasets
!pip install transformers==4.57.3
!pip install peft

In [2]:
from datasets import load_dataset

dataset = load_dataset("mychen76/invoices-and-receipts_ocr_v1")
# Explore one sample
sample = dataset['train'][0]
print(sample.keys())  # ['image', 'ground_truth']

dict_keys(['image', 'id', 'parsed_data', 'raw_data'])


### 2.Split into Train/Val/Test datasets

In [3]:
from datasets import DatasetDict

# Assuming you loaded into `dataset` as above
split_dataset = dataset['train'].train_test_split(test_size=0.15)
train_val = split_dataset['train'].train_test_split(test_size=0.15)

ds = DatasetDict({
    'train': train_val['train'],
    'validation': train_val['test'],
    'test': split_dataset['test']
})

### 3.Organize Data for Multi-modal Model

In [12]:
import json

def format_example(example):
    return {
        'system_message': "Extract all invoice fields and return as JSON",
        'text_input': "",  # or provide further instructions if desired
        'image': example['image'],
        'output': json.dumps(example['raw_data'])
    }

formatted_train = [format_example(x) for x in ds['train']]
formatted_val   = [format_example(x) for x in ds['validation']]  
# Repeat for validation/test

### 4.Load SmolVLM-Instruct from Hugging Face

In [5]:
from transformers import AutoModelForVision2Seq, AutoProcessor

model_id = "HuggingFaceTB/SmolVLM-Instruct"
model = AutoModelForVision2Seq.from_pretrained(model_id)
processor = AutoProcessor.from_pretrained(model_id)

/home/rkuo/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


### 5.Evaluate Base Model (String Matching)

In [6]:
from tqdm import tqdm

def evaluate(model, processor, data, device='cuda'):
    correct = 0
    for item in tqdm(data):
        inputs = processor(
            text=item['system_message'],
            images=item['image'],
            return_tensors="pt"
        ).to(device)
        outputs = model.generate(**inputs, max_new_tokens=256)
        decoded = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        # Crude string matching
        if decoded.strip() == item['output'].strip():
            correct += 1
    return correct / len(data)

### 6. Set Up LoRA Adapters

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],  # update according to model's architecture
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

*(This wraps your base model in a LoRA-adapted model. Adjust target modules as per SmolVLM architecture!)*

### 7.Define SFT Hyperparameters

In [9]:
from trl import SFTConfig

sft_config = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    save_steps=500,
    logging_steps=50,
    eval_steps=200,
#    max_seq_length=512,
    fp16=True,
    push_to_hub=False
)

### 8. Fine-tune Model using SFTTrainer
You’ll use SFTTrainer from trl:

In [16]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=2,
        num_train_epochs=3,
        fp16=True,
    ),
    train_dataset=formatted_train,
    eval_dataset=formatted_val,  # format your val set as above
#    max_seq_length=512,
#    tokenizer=processor,  # SmolVLM uses a processor for multi-modal
    data_collator=None  # customize if needed
)

trainer.train()

KeyError: "Unexpected input keys in examples: ['image']."

*Refer to the attached notebook for exact class signatures and processing!*

### 9. Evaluate Fine-tuned Model

Repeat step 5 (evaluation) but using your fine-tuned weights.

In [ ]:
evaluate(model, processor, data, device='cuda')

### 10. (Optional) Advanced Evaluation
For better measurement, compare the parsed JSON fields with ground truth using field-wise F1 or exact match metrics.

In [ ]:
from sklearn.metrics import f1_score

def eval_json(decoded, target):
    decoded_json = json.loads(decoded)
    target_json = json.loads(target)
    # Compute accuracy/F1 for each field, average, etc.